In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from app.services.reconciliation_service import (ReconciliationService)

class DemoSalesClient:

    def get_sales_records(self):

        return pd.DataFrame([
                {
                    "transaction_id": "101",
                    "invoice_number": "INV-2026-000101",
                    "product_id": "10",
                    "quantity": "5",
                    "status": "validated",
                    "created_at": "2026-08-08T09:00:00Z"
                },
                {
                    "transaction_id": "102",
                    "invoice_number": "INV-2026-000102",
                    "product_id": "11",
                    "quantity": "10",
                    "status": "completed",
                    "created_at": "2026-08-08T10:00:00Z"
                },
                {
                    "transaction_id": "103",
                    "invoice_number": "INV-2026-000103",
                    "product_id": "12",
                    "quantity": "5",
                    "status": "validated",
                    "created_at": "2026-08-08T11:00:00Z"
                },
                {
                    "transaction_id": "104",
                    "invoice_number": "INV-2026-000104",
                    "product_id": "13",
                    "quantity": "3",
                    "status": "validated",
                    "created_at": "2026-08-08T12:00:00Z"
                }])

class DemoInventoryClient:

    def get_inventory_records(self):

        return pd.DataFrame([
                # Exact match
                {
                    "transaction_id": "101",
                    "reservation_id": "501",
                    "batch_id": "20",
                    "reserved_quantity": "5",
                    "status": "reserved",
                    "reserved_at": "2026-08-08T09:01:00Z"
                },
                # Quantity mismatch
                {
                    "transaction_id": "102",
                    "reservation_id": "502",
                    "batch_id": "21",
                    "reserved_quantity": "7",
                    "status": "reserved",
                    "reserved_at": "2026-08-08T10:01:00Z"
                },
                # Duplicate reservations
                {
                    "transaction_id": "103",
                    "reservation_id": "503",
                    "batch_id": "22",
                    "reserved_quantity": "3",
                    "status": "reserved",
                    "reserved_at": "2026-08-08T11:01:00Z"
                },
                {
                    "transaction_id": "103",
                    "reservation_id": "504",
                    "batch_id": "23",
                    "reserved_quantity": "2",
                    "status": "reserved",
                    "reserved_at": "2026-08-08T11:02:00Z"
                },
                # Orphan reservation
                {
                    "transaction_id": "105",
                    "reservation_id": "505",
                    "batch_id": "24",
                    "reserved_quantity": "6",
                    "status": "reserved",
                    "reserved_at": "2026-08-08T13:01:00Z"
                }])

class DemoReportGenerator:

    def __init__(self):

        self.results = None

    def generate_reports(self, results):

        self.results = results

In [3]:
sales_client = DemoSalesClient()
inventory_client = DemoInventoryClient()
report_generator = DemoReportGenerator()

service = ReconciliationService(
    sales_client=sales_client,
    inventory_client=inventory_client,
    report_generator=report_generator)

In [4]:
result = service.run()

result.keys()

dict_keys(['source_df', 'target_df', 'comparison_results', 'comparison_df', 'mismatches', 'fuzzy_matches', 'summary_df'])

In [5]:
display(result["source_df"])

,transaction_id,invoice_number,product_id,quantity,status,created_at
0,101,INV-2026-000101,10,5,VALIDATED,2026-08-08 09:00:00+00:00
1,102,INV-2026-000102,11,10,COMPLETED,2026-08-08 10:00:00+00:00
2,103,INV-2026-000103,12,5,VALIDATED,2026-08-08 11:00:00+00:00
3,104,INV-2026-000104,13,3,VALIDATED,2026-08-08 12:00:00+00:00


In [6]:
display(result["target_df"])

,transaction_id,reservation_id,batch_id,reserved_quantity,status,reserved_at
0,101,501,20,5,RESERVED,2026-08-08 09:01:00+00:00
1,102,502,21,7,RESERVED,2026-08-08 10:01:00+00:00
2,103,503,22,3,RESERVED,2026-08-08 11:01:00+00:00
3,103,504,23,2,RESERVED,2026-08-08 11:02:00+00:00
4,105,505,24,6,RESERVED,2026-08-08 13:01:00+00:00


In [7]:
final_results = pd.DataFrame(result["comparison_results"])

display(final_results)

,transaction_id,invoice_number,status,mismatch_type,sales_status,inventory_status,quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,reservations,sales_quantity,details
0,101,INV-2026-000101,MATCHED,NaN,VALIDATED,RESERVED,5.0,5.0,1,501,20,[501],[20],[5],[RESERVED],[2026-08-08 09:01:00+00:00],"[{'reservation_id': '501', 'batch_id': '20', '...",NaN,NaN
1,102,INV-2026-000102,MISMATCHED,QUANTITY_MISMATCH,COMPLETED,RESERVED,10.0,7.0,1,502,21,[502],[21],[7],[RESERVED],[2026-08-08 10:01:00+00:00],"[{'reservation_id': '502', 'batch_id': '21', '...",10.0,Sales transaction quantity does not match Inve...
2,103,NaN,MISMATCHED,DUPLICATE_RESERVATION,VALIDATED,"[RESERVED, RESERVED]",5.0,5.0,2,NaN,NaN,"[503, 504]","[22, 23]","[3, 2]","[RESERVED, RESERVED]","[2026-08-08 11:01:00+00:00, 2026-08-08 11:02:0...","[{'reservation_id': '503', 'batch_id': '22', '...",NaN,Multiple Inventory reservations exist for the ...
3,104,INV-2026-000104,MISMATCHED,MISSING_RESERVATION,VALIDATED,None,3.0,NaN,0,NaN,NaN,[],[],[],[],[],NaN,3.0,Sales transaction has no corresponding Invento...
4,105,NaN,MISMATCHED,ORPHAN_RESERVATION,NaN,[RESERVED],NaN,6.0,1,NaN,NaN,[505],[24],[6],[RESERVED],[2026-08-08 13:01:00+00:00],"[{'reservation_id': '505', 'batch_id': '24', '...",NaN,Inventory reservation references a transaction...


In [8]:
print(final_results[
        [
            "transaction_id",
            "status",
            "mismatch_type"
        ]].to_string(index=False))

 transaction_id     status         mismatch_type
            101    MATCHED                   NaN
            102 MISMATCHED     QUANTITY_MISMATCH
            103 MISMATCHED DUPLICATE_RESERVATION
            104 MISMATCHED   MISSING_RESERVATION
            105 MISMATCHED    ORPHAN_RESERVATION


In [9]:
duplicate_result = final_results[final_results["transaction_id"] == 103].iloc[0]

print("Mismatch type:", duplicate_result["mismatch_type"])

print("Reservation IDs:", duplicate_result["reservation_ids"])

print("Reserved quantities:", duplicate_result["reserved_quantities"])

print("Reservation count:", duplicate_result["reservation_count"])

Mismatch type: DUPLICATE_RESERVATION
Reservation IDs: ['503', '504']
Reserved quantities: [3, 2]
Reservation count: 2


In [10]:
display(pd.DataFrame(result["mismatches"]))

,transaction_id,mismatch_type,invoice_number,sales_quantity,reserved_quantity,reservation_count,reservation_id,batch_id,reservation_ids,batch_ids,reserved_quantities,reservation_statuses,reservation_timestamps,details,reservations
0,104,MISSING_RESERVATION,INV-2026-000104,3.0,NaN,0,NaN,NaN,[],[],[],[],[],Sales transaction has no corresponding Invento...,NaN
1,103,DUPLICATE_RESERVATION,NaN,NaN,5.0,2,NaN,NaN,"[503, 504]","[22, 23]","[3, 2]","[RESERVED, RESERVED]","[2026-08-08 11:01:00+00:00, 2026-08-08 11:02:0...",Multiple Inventory reservations exist for the ...,"[{'reservation_id': '503', 'batch_id': '22', '..."
2,102,QUANTITY_MISMATCH,INV-2026-000102,10.0,7.0,1,502,21,[502],[21],[7],[RESERVED],[2026-08-08 10:01:00+00:00],Sales transaction quantity does not match Inve...,"[{'reservation_id': '502', 'batch_id': '21', '..."
3,105,ORPHAN_RESERVATION,NaN,NaN,6.0,1,NaN,NaN,[505],[24],[6],[RESERVED],[2026-08-08 13:01:00+00:00],Inventory reservation references a transaction...,"[{'reservation_id': '505', 'batch_id': '24', '..."


In [11]:
display(result["summary_df"])

,category_type,category,count
0,status,MISMATCHED,4
1,status,MATCHED,1
2,mismatch_type,QUANTITY_MISMATCH,1
3,mismatch_type,DUPLICATE_RESERVATION,1
4,mismatch_type,MISSING_RESERVATION,1
5,mismatch_type,ORPHAN_RESERVATION,1


In [12]:
from pathlib import Path

REPORT_DIR = Path("reports")

for report in sorted(REPORT_DIR.glob("*.csv")):
    
    print(report.name)

summary.csv


In [13]:
from pathlib import Path

REPORT_DIR = Path("reports")

for path in sorted(REPORT_DIR.rglob("*")):
    
    print(path)

reports\charts
reports\charts\reconciliation_summary.png
reports\summary.csv
